
## DATA READING

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df = spark.read.format("parquet")\
          .option("inferSchema", True)\
          .load("abfss://bronze@adlscarproject.dfs.core.windows.net/raw_data")

In [0]:
display(df)

In [0]:
df = df.withColumn("Model_cat", split(col("Model_ID"), '-')[0])
display(df)

In [0]:
df.withColumn('Units_Sold', col('Units_Sold').cast(StringType())).display()

In [0]:
df = df.withColumn('Revenue_per_unit', col("Revenue")/col("Units_Sold"))
display(df)

In [0]:

display(df.groupBy('Year', 'BranchName')\
    .agg(sum("Units_Sold").alias('Total_Units'))\
    .orderBy('Year', 'Total_Units', ascending=[True, False]))



Databricks visualization. Run in Databricks to view.


## DATA WRITING

In [0]:
df.write.format('parquet')\
    .mode('append')\
    .option("path", "abfss://silver@adlscarproject.dfs.core.windows.net/car_sales")\
    .save()


In [0]:
%sql
-- Fetching / Querying data in Datalake

SELECT * FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`